In [1]:
pip install groq langchain-groq langgraph langchain-core langchain-community

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Replace 'gsk_your_actual_key_here' with the key you copied from Groq
groq_key = "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"

with open(".env", "w") as f:
    f.write(f'GROQ_API_KEY="{groq_key}"\n')

print(".env file successfully created in your project folder!")

.env file successfully created in your project folder!


In [7]:
import os
from dotenv import load_dotenv
from groq import Groq

# 1. Load environment variables
load_dotenv()

# 2. Connect Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 3. Fetch active models catalog
models_list = client.models.list()

print("--- Available Active Groq Models ---")
for model in models_list.data:
    print(f"- {model.id}")

--- Available Active Groq Models ---
- openai/gpt-oss-120b
- qwen/qwen3.8-27b
- meta-llama/llama-prompt-guard-2-22m
- whisper-large-v3-turbo
- canopylabs/orpheus-v1-english
- meta-llama/llama-prompt-guard-2-86m
- openai/gpt-oss-20b
- whisper-large-v3
- allam-2-7b
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-arabic-saudi


In [8]:
import os
from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq

# 1. Load environment variables
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if api_key and api_key.startswith("gsk_"):
    print("GROQ_API_KEY successfully loaded from .env!")
    
    # 2. Fetch available models dynamically
    raw_client = Groq(api_key=api_key)
    available_models = [m.id for m in raw_client.models.list().data if "llama" in m.id]
    
    # Choose the first available Llama model or default to standard llama3-8b-8192
    selected_model = available_models[0] if available_models else "llama3-8b-8192"
    print(f"Auto-selected Active Model: '{selected_model}'")
    
    # 3. Initialize LangChain ChatGroq with active model
    llm = ChatGroq(
        model_name=selected_model,
        groq_api_key=api_key,
        temperature=0.0
    )
    
    # 4. Test API connection
    response = llm.invoke("Say 'Groq connection successful!'")
    print("API Response:", response.content)

else:
    print("Error: GROQ_API_KEY is missing or invalid in .env file.")

GROQ_API_KEY successfully loaded from .env!
Auto-selected Active Model: 'meta-llama/llama-prompt-guard-2-22m'
API Response: 0.0007253962685354054


In [10]:
import os
from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq

# 1. Load API key from environment
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

raw_client = Groq(api_key=api_key)

# Fetch all active models
all_models = raw_client.models.list().data

# Print available model IDs for visibility
print("--- Currently Available Groq Models ---")
for m in all_models:
    print(f"- {m.id}")

# Recommended default production models on Groq
candidate_models = ["llama-3.3-70b-versatile", "llama-3.2-3b-preview", "llama-3.1-8b-instant", "mixtral-8x7b-32768"]

working_model = None

for model_id in candidate_models:
    try:
        test_llm = ChatGroq(
            model_name=model_id,
            groq_api_key=api_key,
            temperature=0.0
        )
        # Test call
        res = test_llm.invoke("Hi")
        working_model = model_id
        print(f"\nSUCCESS: Connected to active model '{working_model}'!")
        break
    except Exception as e:
        continue

if working_model:
    llm = ChatGroq(
        model_name=working_model,
        groq_api_key=api_key,
        temperature=0.0
    )
    response = llm.invoke("Say 'Groq Chat Model is ready for multi-agent execution!'")
    print("API Text Response:", response.content)
else:
    print("No working model found. Please check your Groq console dashboard.")

--- Currently Available Groq Models ---
- canopylabs/orpheus-arabic-saudi
- canopylabs/orpheus-v1-english
- whisper-large-v3
- openai/gpt-oss-20b
- openai/gpt-oss-safeguard-20b
- whisper-large-v3-turbo
- meta-llama/llama-prompt-guard-2-86m
- allam-2-7b
- qwen/qwen3.8-27b
- meta-llama/llama-prompt-guard-2-22m
- openai/gpt-oss-120b
No working model found. Please check your Groq console dashboard.


In [11]:
import os
from dotenv import load_dotenv
from groq import Groq
from langchain_groq import ChatGroq

# 1. Load API Key from .env
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

raw_client = Groq(api_key=api_key)

# 2. Fetch all active models from your Groq account
all_models = [m.id for m in raw_client.models.list().data]

# 3. Filter out audio (whisper) and security (guard/safeguard) models
chat_candidates = [
    m for m in all_models 
    if not any(excluded in m.lower() for excluded in ["whisper", "guard", "safeguard", "arabic"])
]

print("Detected Text Chat Models:", chat_candidates)

working_model = None

# 4. Loop through available text models and find the first working one
for model_id in chat_candidates:
    try:
        print(f"Testing connection with: '{model_id}'...")
        test_llm = ChatGroq(
            model_name=model_id,
            groq_api_key=api_key,
            temperature=0.0
        )
        res = test_llm.invoke("Say 'Groq connection successful!'")
        working_model = model_id
        print(f"\nSUCCESS! Locked working model: '{working_model}'")
        print("API Response:", res.content)
        break
    except Exception as e:
        print(f"Skipping {model_id}: {e}")
        continue

Detected Text Chat Models: ['openai/gpt-oss-20b', 'qwen/qwen3.8-27b', 'allam-2-7b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-120b']
Testing connection with: 'openai/gpt-oss-20b'...

SUCCESS! Locked working model: 'openai/gpt-oss-20b'
API Response: Groq connection successful!
